# Indian Number Plate OCR - Fine-tune PaddleOCR (PP-OCRv3 mobile rec)

Steps:
1. Runtime > Change runtime type > GPU (T4 is fine), then run cells top to bottom.
2. When cell 3 asks for a file, upload `release.zip` (zip of the `ocr_training/release/` folder from the anpr-ai-service repo - contains `crops/`, `train_list.txt`, `val_list.txt`, `en_dict.txt`, `train_config.yml`).
3. At the end you'll get two downloads:
   - `indian_plate_rec_v2_inference.zip` - unzip into `weights/rec_indian_plate_infer/` in the repo, replacing the old files. This is what `main.py` actually loads.
   - `indian_plate_rec_v2_checkpoint.zip` - the trainable checkpoint (weights + optimizer state). Keep this one aside (e.g. in `ocr_training/checkpoints/`) - it lets a *future* training round resume from here with just the new data, instead of starting from the generic pretrained model and re-merging everything collected so far.

In [ ]:
!nvidia-smi

In [ ]:
import os
if not os.path.isdir('/content/PaddleOCR'):
    !git clone https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR
%cd /content/PaddleOCR
!pip install -q -r requirements.txt

# paddlepaddle-gpu wheels aren't on plain PyPI - they're hosted on PaddlePaddle's
# own index, keyed by CUDA version. Try Colab's current CUDA build first, fall
# back to older tags if this particular Colab image ships an older driver.
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ \
  || pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu123/ \
  || pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu121/ \
  || pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu118/

!python -c "import paddle; paddle.utils.run_check()"

In [ ]:
import shutil
from google.colab import files
print("Upload release.zip now:")
uploaded = files.upload()
for fname in uploaded.keys():
    shutil.move(fname, '/content/release.zip')
print("Saved to /content/release.zip")

In [ ]:
!mkdir -p /content/release /content/PaddleOCR/train_data /content/PaddleOCR/pretrain_models
!unzip -q -o /content/release.zip -d /content/release
!cp -r /content/release/crops /content/release/train_list.txt /content/release/val_list.txt /content/release/en_dict.txt /content/PaddleOCR/train_data/
!cp /content/release/train_config.yml /content/PaddleOCR/train_config.yml
!echo 'train:' && wc -l /content/PaddleOCR/train_data/train_list.txt
!echo 'val:' && wc -l /content/PaddleOCR/train_data/val_list.txt
!echo 'crops:' && ls /content/PaddleOCR/train_data/crops | wc -l

In [ ]:
%cd /content/PaddleOCR
!wget -nc -P ./pretrain_models/ https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_rec_train.tar
!tar -xf ./pretrain_models/en_PP-OCRv3_rec_train.tar -C ./pretrain_models/

## Train

`train_config.yml` already points `Global.pretrained_model` at the extracted checkpoint and turns up `RecAug`'s angle/blur/brightness probabilities. 100 epochs on ~2.3k images typically takes well under an hour on a Colab T4 - watch the printed `acc` on the eval lines; stop early (Runtime > Interrupt) once it plateaus, the latest checkpoint under `output/indian_plate_rec_v2/` is still usable.

In [ ]:
!python3 tools/train.py -c train_config.yml

## Export to inference format

Uses `best_accuracy` (the checkpoint with the highest eval accuracy PaddleOCR saved during training, not necessarily the last epoch).

In [ ]:
!python3 tools/export_model.py -c train_config.yml -o Global.pretrained_model=./output/indian_plate_rec_v2/best_accuracy Global.save_inference_dir=./inference/indian_plate_rec_v2/

In [ ]:
%cd /content/PaddleOCR/inference/indian_plate_rec_v2
!ls -la
!zip -q -r /content/indian_plate_rec_v2_inference.zip .
from google.colab import files
files.download('/content/indian_plate_rec_v2_inference.zip')

## Download the training checkpoint too

Not just the inference export - `output/indian_plate_rec_v2/` holds the actual trainable checkpoint (`best_accuracy.pdparams` + optimizer state). Keep this zip alongside the inference one so a *future* round can resume training from exactly this point instead of starting from the generic pretrained model again - true incremental fine-tuning on just the new data, rather than needing to re-merge and re-train on everything collected so far.

In [ ]:
%cd /content/PaddleOCR
!zip -q -r /content/indian_plate_rec_v2_checkpoint.zip output/indian_plate_rec_v2 train_config.yml -x "output/indian_plate_rec_v2/train.log"
!ls -la /content/*.zip
from google.colab import files
files.download('/content/indian_plate_rec_v2_checkpoint.zip')